# Auditoría de conteos — población perinatal NCHS

Etapa del framework: **E4** (auditoría de conteos) y puerta **P1**. Lee `data/interim/perinatal_<año>.parquet` (adaptador de datos) y produce solo conteos agregados en `data/outputs_02_count_audit.txt`; los conteos de 1 a 9 se muestran como "<10". Las frecuencias de los predictores se calculan en toda la población, nunca separadas por desenlace; la única comparación por componente (Q-C7) es una verificación de fuga de la puerta P2. Versión en inglés: `02_count_audit.ipynb` (mismo código). Tarda unos 3 minutos y usa unos 2,5 GB de memoria.

In [ ]:
import sys, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from libraries.perinatal_builder import YEARS, SHARED, FETAL_DIR, read_fetal
from libraries.nchs_reader import OUT_DIR

OUT_FILE = ROOT / "data" / "outputs_02_count_audit.txt"
# Gate G1: reference counts of the data adapter (data/outputs_01_build_perinatal.txt)
EXPECTED = {2016: (3956689, 27006), 2017: (3866265, 25929), 2018: (3802569, 25324),
            2019: (3757755, 24173), 2020: (3623630, 23261), 2021: (3674138, 23364),
            2022: (3677156, 22780), 2023: (3605074, 22337)}
# "Unknown or not stated" codes (user guides); character fields use "U"
UNKNOWN = {"MRACEHISP": [8], "MEDUC": [9], "MBSTATE_REC": [3], "PRIORLIVE": [99], "PRIORDEAD": [99],
           "ILLB_R": [999], "PRECARE": [99], "RF_CESARN": [99], "CIG_0": [99]}
CATEGORICAL = ["MRACEHISP", "MEDUC", "MBSTATE_REC", "WIC", "CIG_REC", "RF_PDIAB", "RF_GDIAB",
               "RF_PHYPE", "RF_GHYPE", "RF_EHYPE", "RF_INFT", "RF_CESAR", "DPLURAL"]
NUMERIC = ["MAGER", "BMI", "PRECARE", "PRIORLIVE", "PRIORDEAD", "ILLB_R", "RF_CESARN", "CIG_0"]
FIELDS = CATEGORICAL + NUMERIC
lines = []

def show(title, obj):
    """Print and keep an aggregate table; counts from 1 to 9 are suppressed as '<10'."""
    if isinstance(obj, pd.DataFrame) or isinstance(obj, pd.Series):
        txt = obj.map(lambda v: "<10" if isinstance(v, (int, np.integer)) and 0 < v < 10 else v) \
            if isinstance(obj, pd.DataFrame) else obj
        txt = txt.to_string()
    else:
        txt = str(obj)
    block = f"\n== {title}\n{txt}"
    print(block); lines.append(block)

def is_unknown(c, s):
    if c == "BMI":
        return s.ge(99) | s.isna()
    if c in UNKNOWN:
        return s.isin(UNKNOWN[c]).fillna(True) | s.isna()
    if c in ("MAGER", "DPLURAL"):
        return s.isna()
    return s.astype("string").eq("U").fillna(True) | s.isna()

## 1. Flujo de la población, eventos, momento de la muerte fetal y puerta P1

In [ ]:
# Q-C1 flow per year and component, Q-C2 events, Q-C3 timing of fetal death; G1 check
flow, events, estofd, unk_year, unk_comp, n_comp = [], [], [], [], {}, {}
cat_counts, num_counts = {c: {} for c in CATEGORICAL}, {c: {} for c in NUMERIC}
for y in YEARS:
    d = pd.read_parquet(OUT_DIR / f"perinatal_{y}.parquet")
    for comp, g in d.groupby("component", observed=True):
        ga = g["ga_weeks"]
        us = g["RESTATUS"] < 4
        flow.append({"year": y, "component": comp, "records": len(g),
                     "foreign_resident": int((~us).sum()),
                     "ga_unknown": int((us & (ga.isna() | (ga == 99))).sum()),
                     "ga_under_22": int((us & (ga < 22)).sum()),
                     "ga_over_47": int((us & ga.between(48, 98)).sum()),
                     "population": int(g["in_population"].sum())})
    p = d[d["in_population"]]
    ge28 = p["ga_weeks"] >= 28
    events.append({"year": y, "products": len(p), "events": int(p["label_main"].sum()),
                   "fetal": int(p["label_fetal"].sum()), "early_neonatal": int(p["label_early_neonatal"].sum()),
                   "neonatal_first_24h": int(p["died_first_24h"].sum()),
                   "products_ge28": int(ge28.sum()), "events_ge28": int((ge28 & p["label_main"]).sum())})
    f = p[p["component"] == "fetal_death"]["estofd"].fillna("blank").value_counts()
    estofd.append({"year": y, **{k: int(v) for k, v in f.items()}})
    row = {"year": y}
    for c in FIELDS:
        row[c] = round(100 * is_unknown(c, p[c]).mean(), 1)
        for comp in ("live_birth", "fetal_death"):
            q = p.loc[p["component"] == comp, c]
            unk_comp[(c, comp)] = unk_comp.get((c, comp), 0) + int(is_unknown(c, q).sum())
        if c in CATEGORICAL:
            for k, v in p[c].astype("string").fillna("missing").value_counts().items():
                cat_counts[c][k] = cat_counts[c].get(k, 0) + int(v)
        else:
            vals = p[c].astype("Float64").round(1)
            for k, v in vals.value_counts(dropna=False).items():
                num_counts[c][k] = num_counts[c].get(k, 0) + int(v)
    unk_year.append(row)
    for comp in ("live_birth", "fetal_death"):
        n_comp[comp] = n_comp.get(comp, 0) + int((p["component"] == comp).sum())
    del d, p

ev = pd.DataFrame(events).set_index("year")
ev["per_1000"] = (1000 * ev["events"] / ev["products"]).round(2)
ev["pct_fetal"] = (100 * ev["fetal"] / ev["events"]).round(1)
show("Q-C1 Population flow by year and component", pd.DataFrame(flow).set_index(["year", "component"]))
show("Q-C2 Events by year (population: 22-47 weeks, US residents)", ev)
est = pd.DataFrame(estofd).set_index("year").fillna(0).astype(int)
show("Q-C3 Estimated time of fetal death (N before labor, L/A during labor, U unknown)", est)

checks = {y: (int(ev.loc[y, "products"]), int(ev.loc[y, "events"])) == EXPECTED[y] for y in YEARS}
show("Gate G1: reference counts reproduced", pd.Series(checks))
assert all(checks.values()), "population or events differ from the data adapter"

## 2. Frecuencias de los predictores candidatos, faltantes por año y verificación de fuga

In [ ]:
# Q-C4 frequencies of every candidate (whole population, never split by outcome)
total = int(ev["products"].sum())
for c in CATEGORICAL:
    s = pd.Series(cat_counts[c]).sort_index()
    show(f"Q-C4 {c}: products and % of the population", pd.DataFrame({"n": s, "pct": (100 * s / total).round(2)}))

def quantiles(counts, exclude):
    s = pd.Series(counts)
    s = s[[k for k in s.index if pd.notna(k) and k not in exclude]].sort_index()
    cum = s.cumsum() / s.sum()
    return {q: float(s.index[np.searchsorted(cum.values, q)]) for q in (0.05, 0.25, 0.5, 0.75, 0.95)}

rows = []
for c in NUMERIC:
    excl = {99.9} if c == "BMI" else set(UNKNOWN.get(c, [])) | ({888} if c == "ILLB_R" else set())
    q = quantiles(num_counts[c], excl)
    unk = sum(v for k, v in num_counts[c].items() if pd.isna(k) or k in excl - {888} or (c == "BMI" and k >= 99))
    row = {"field": c, "pct_unknown": round(100 * unk / total, 2), **{f"p{int(k*100):02d}": v for k, v in q.items()}}
    if c == "ILLB_R":
        row["pct_first_birth_888"] = round(100 * num_counts[c].get(888, 0) / total, 2)
    rows.append(row)
show("Q-C5 Numeric candidates: unknown and percentiles (unknown codes excluded)", pd.DataFrame(rows).set_index("field"))

# Q-C6 unknown by year (2016-2017: not every state used the revised fetal death report)
show("Q-C6 % unknown by year, whole population", pd.DataFrame(unk_year).set_index("year").T)

# Q-C7 leakage check (gate G2): is "unknown" a marker of the type of certificate?
show("Q-C7 % unknown by component (live-birth certificate vs fetal death report)", pd.DataFrame({
    "live_births": {c: round(100 * unk_comp[(c, "live_birth")] / n_comp["live_birth"], 1) for c in FIELDS},
    "fetal_deaths": {c: round(100 * unk_comp[(c, "fetal_death")] / n_comp["fetal_death"], 1) for c in FIELDS}}))

## 3. Edad gestacional del componente fetal y salida agregada

In [ ]:
# Q-C8 gestational age of the fetal component: obstetric estimate vs combined gestation
rows = []
for y in YEARS:
    fd = read_fetal(y)
    fd = fd[(fd["RESTATUS"] < 4)]
    oe, cg = fd["OEGEST_COMB"], fd["COMBGEST"]
    both = oe.between(17, 47) & cg.between(17, 47)
    rows.append({"year": y, "records_us": len(fd), "both_known": int(both.sum()),
                 "same_week_pct": round(100 * (oe[both] == cg[both]).mean(), 1),
                 "within_1_week_pct": round(100 * ((oe[both] - cg[both]).abs() <= 1).mean(), 1),
                 "ge22_by_oe": int(oe.between(22, 47).sum()), "ge22_by_combgest": int(cg.between(22, 47).sum())})
show("Q-C8 Fetal deaths: obstetric estimate (used) vs combined gestation", pd.DataFrame(rows).set_index("year"))

OUT_FILE.write_text("Count audit of the perinatal population (aggregate counts only; 1-9 shown as <10)\n"
                    + "\n".join(lines) + "\n", encoding="utf-8")